# 🔐  접근 제어 (public / protected / private)


---

## 🎯 학습 목표

- **캡슐화(encapsulation)** 가 왜 필요한지 실제 사고 사례로 설명할 수 있다.
- `name`, `_name`, `__name` 세 가지 표기의 차이를 구분할 수 있다.
- **이름 맹글링(name mangling)** 이 무엇인지 직접 확인할 수 있다.
- getter/setter 와 `@property` 로 **안전하게 값을 읽고 쓰는 통로**를 만들 수 있다.
- AI 서비스의 **API 키를 숨기는 클래스**를 직접 만들 수 있다.



---

## 0. 왜 접근 제어가 필요한가요?

### 💸 사고 사례: 밖에서 잔액을 마음대로 바꿀 수 있다면?

지난 시간에 만든 통장 클래스에는 **구멍**이 있었습니다. 아래 코드를 보세요.

In [ ]:
# ✍️ 아래 코드를 실행하고, 무엇이 문제인지 한 줄로 적어 보세요.
class BankAccount:
    def __init__(self, owner, balance=0):
        self.balance = balance

acc = BankAccount("하늘", 10000)
acc.balance = 1000000000
print("잔액:", acc.balance)

acc.balance = -50000
print("잔액:", acc.balance)

# 📌 무엇이 문제인가요?
#    잔액(balance)이 public 이라 클래스 밖에서 아무 검사 없이 10억이나 음수(-50000)로 마음대로 바꿀 수 있다.

### 🏪 비유: 편의점 구조

| 공간 | 누가 들어갈 수 있나 | 파이썬 표기 |
|---|---|---|
| **진열대** 🥤 | 손님 누구나 | `name` (public) |
| **창고** 📦 | 직원과 신입 직원(상속받은 자식)만 | `_name` (protected) |
| **금고** 🔒 | 점장(클래스 자신)만 | `__name` (private) |

- 손님이 창고에 들어가 재고를 마음대로 바꾸면 가게가 망하죠.
- **중요한 데이터는 안쪽에 숨기고, 정해진 창구(메서드)로만 처리**하는 것이 **캡슐화**입니다.

### 💊 비유 ②: 약 포장

- 알약은 **캡슐**에 싸여 있습니다. 안에 뭐가 들었는지 몰라도 **먹는 방법(복용법)** 만 알면 됩니다.
- 클래스도 마찬가지입니다. 내부 구조는 숨기고 **사용법(메서드)** 만 열어 둡니다.

### ⚠️ 

> **파이썬에는 Java 나 C++ 처럼 완전히 막는 `private` 키워드가 없습니다.**
> 대신 **"약속(convention)"** 과 **"이름 바꾸기(name mangling)"** 로 표현합니다.
> 그래서 파이썬에서는 이렇게 말해요: **"우리는 모두 성인이다 (We are all consenting adults)"**
> → 문법으로 완전히 막지 않고, **"여기는 건드리지 마세요"라는 신호를 주는 것**이 목적입니다.

---

## 1. public: 누구나 접근 가능 (밑줄 없음)

### 📖 개념

- 밑줄 없이 쓰는 평범한 속성과 메서드는 모두 **public** 입니다.
- 클래스 밖에서 자유롭게 읽고 쓸 수 있습니다.
- **밖에서 알아도 되고 바뀌어도 괜찮은 정보**에 씁니다. (예: 상품명, 사용자 닉네임)

In [ ]:
# ✍️ 빈칸을 채우세요
class Product:
    def __init__(self, name, price):
        self.name = name      # public
        self.price = price    # public

    def info(self):           # public 메서드
        print(f"{self.name}: {self.price}원")


p = Product("콜라", 1500)
p.info()

# 밖에서 자유롭게 읽고 쓸 수 있음
print(p.name)
p.name = "제로콜라"
p.info()

---

## 2. protected: 밑줄 1개 `_name` — "직원 전용" 표지판

### 📖 개념

- 이름 앞에 **밑줄 1개**를 붙입니다: `self._balance`
- 의미: **"클래스 내부와 상속받은 자식 클래스에서만 쓰세요"**
- ⚠️ **문법으로 막지는 않습니다.** 밖에서 접근하면 **그냥 됩니다.**
- 즉 **"직원 전용" 표지판**과 같습니다. 손님이 밀고 들어갈 수는 있지만, 들어가면 안 되는 곳이죠.

### 🚪 비유

- 식당의 **"Staff Only"** 문: 잠겨 있진 않지만 손님은 열지 않습니다.
- 팀 프로젝트에서 `_` 를 본 동료는 **"이건 내부용이니 직접 건드리지 말자"** 로 이해합니다.

In [ ]:
# ✍️ 직접 작성해 보세요
# 1) Employee 클래스: __init__ 에서 name(public), _salary(protected) 저장
# 2) show() 메서드: "이름: 연봉 N만원" 출력
# 3) 객체를 만들어 show() 호출
# 4) 밖에서 e._salary 를 print → 에러가 나는지 안 나는지 직접 확인!
class Employee:
    def __init__(self, name, salary):
        self.name = name        # public
        self._salary = salary   # protected

    def show(self):
        print(f"{self.name}: 연봉 {self._salary}만원")


e = Employee("민수", 4000)
e.show()

print(e._salary)   # 에러 안 남! → _ 는 "건드리지 마세요"라는 약속일 뿐 문법으로 막지 않음

### 🧬 protected 는 상속과 함께 쓸 때 의미가 살아납니다 (다음 차시 맛보기)

자식 클래스는 부모의 `_속성` 을 **자유롭게 쓸 수 있습니다.** 이게 protected 의 존재 이유입니다.

In [ ]:
# ✍️ 상속 맛보기 (다음 차시에 자세히 배웁니다)
# 1) Employee 클래스 (name, _salary)
# 2) Manager(Employee) 클래스를 만들고, show_bonus() 에서
#    부모의 self._salary 를 사용해 "이름 보너스: N만원" 출력 (보너스 = 연봉의 10%)
# 3) Manager 객체를 만들어 show_bonus() 호출
class Employee:
    def __init__(self, name, salary):
        self.name = name
        self._salary = salary


class Manager(Employee):
    def show_bonus(self):
        bonus = self._salary * 0.1   # 자식 클래스는 부모의 _salary 를 자유롭게 사용
        print(f"{self.name} 보너스: {bonus:.0f}만원")


m = Manager("지현", 6000)
m.show_bonus()

---

## 3. private: 밑줄 2개 `__name` — 금고에 넣기

### 📖 개념

- 이름 앞에 **밑줄 2개**를 붙입니다: `self.__balance`
- 의미: **"이 클래스 안에서만 쓰세요"**
- 밖에서 `객체.__balance` 로 접근하면 **`AttributeError` 가 발생합니다!**
- 단, 완전히 사라진 건 아닙니다. 파이썬이 **이름을 몰래 바꿔 놓은 것**입니다. (→ 다음 절에서 확인)

### ⚠️ 주의: 밑줄 2개가 **앞에만** 있어야 합니다

| 표기 | 결과 |
|---|---|
| `__balance` | ✅ private (이름이 바뀜) |
| `__init__` | ❌ 앞뒤로 2개 → 파이썬 **특별 메서드**, private 아님 |
| `_balance` | protected |

In [ ]:
# ✍️ 빈칸을 채우세요
class BankAccount:
    def __init__(self, owner, balance=0):
        self.owner = owner        # public
        self.__balance = balance  # private

    def show(self):
        print(f"{self.owner} 잔액: {self.__balance}원")   # 클래스 안에서는 정상 접근


acc = BankAccount("하늘", 10000)
acc.show()

# 밖에서 접근하면 어떤 에러가 날까요? 예측한 뒤 실행!
# 예측: AttributeError
try:
    print(acc.__balance)
except AttributeError as e:
    print("❌ AttributeError:", e)

### 🔍 이름 맹글링(name mangling): 파이썬의 숨기기 마법

파이썬은 `__balance` 를 내부적으로 **`_클래스이름__balance`** 로 **이름을 바꿔 버립니다.**

```
__balance  →  _BankAccount__balance
```

### 🏷️ 비유

- 금고에 넣은 게 아니라, **라벨을 몰래 다른 이름으로 바꿔 붙인 것**입니다.
- 원래 라벨(`__balance`)로 찾으면 "그런 거 없다"고 하지만, 바뀐 라벨을 알면 찾을 수 있습니다.
- 그래서 파이썬의 private 은 **"완전한 차단"이 아니라 "실수 방지 + 강한 신호"** 입니다.

In [ ]:
# ✍️ 이름 맹글링을 직접 확인해 보세요
# 1) acc.__dict__ 를 출력 → 실제 속성 이름이 무엇으로 바뀌었는지 보기
# 2) 바뀐 이름(_클래스이름__속성)으로 잔액에 접근해 출력해 보기
# 📌 발견한 실제 속성 이름: _BankAccount__balance
print(acc.__dict__)                # {'owner': '하늘', '_BankAccount__balance': 10000}
print(acc._BankAccount__balance)   # 10000 → 바뀐 이름으로는 접근됨 (실무에서는 금지!)

> 💡 **그럼 private 은 의미가 없는 걸까요?**
> 아닙니다! `_BankAccount__balance` 처럼 **일부러 길고 이상하게 써야만** 접근됩니다.
> → **실수로 건드리는 일을 막아 주고**, 동료에게 **"절대 건드리지 말라"는 가장 강한 신호**가 됩니다.
> 실무에서 이렇게 우회해서 쓰면 **코드 리뷰에서 반드시 지적받습니다.**

---

## 4. 그럼 값을 어떻게 읽고 바꾸나요? → getter / setter

### 📖 개념

- private 으로 **숨긴 뒤**, 정해진 **메서드(창구)** 로만 읽고 쓰게 합니다.
- **getter**: 값을 **읽는** 메서드 (`get_balance`)
- **setter**: 값을 **바꾸는** 메서드 (`set_balance`, `deposit`, `withdraw`)
- 핵심 장점: **값을 바꿀 때 검사(validation)를 넣을 수 있습니다.**

### 🏦 비유: 은행 창구

- 금고(`__balance`)에 손님이 직접 들어갈 수는 없습니다.
- **창구 직원(메서드)** 에게 요청하면, 직원이 **신분 확인과 잔액 검사**를 하고 처리합니다.

In [ ]:
# ✍️ 빈칸을 채우세요
class BankAccount:
    def __init__(self, owner, balance=0):
        self.owner = owner
        self.__balance = balance

    # getter: 값 읽기
    def get_balance(self):
        return self.__balance

    # setter: 값 바꾸기 (검사 포함)
    def set_balance(self, amount):
        if amount < 0:
            print("⚠️ 잔액은 음수가 될 수 없습니다")
            return
        self.__balance = amount

    def deposit(self, amount):
        if amount <= 0:
            print("⚠️ 입금액은 0보다 커야 합니다")
            return
        self.__balance += amount
        print(f"{amount}원 입금 → 잔액 {self.__balance}원")

    def withdraw(self, amount):
        if amount <= 0:
            print("⚠️ 출금액은 0보다 커야 합니다")
            return
        if amount > self.__balance:
            print(f"⚠️ 잔액 부족 (잔액 {self.__balance}원)")
            return
        self.__balance -= amount
        print(f"{amount}원 출금 → 잔액 {self.__balance}원")


acc = BankAccount("하늘", 10000)
print("잔액:", acc.get_balance())

acc.deposit(5000)
acc.withdraw(30000)    # 잔액 부족 → 거부
acc.withdraw(3000)
acc.set_balance(-50000)  # 음수 → 거부
print("최종 잔액:", acc.get_balance())

### 🎁 한 걸음 더: `@property` 

getter 를 쓸 때마다 `acc.get_balance()` 처럼 괄호를 붙이는 게 번거롭죠.
`@property` 를 쓰면 **메서드를 속성처럼** 쓸 수 있습니다.

In [ ]:
# ✍️ @property 를 직접 써 보세요
# 1) BankAccount 클래스에 owner(public), __balance(private) 저장
# 2) @property 를 붙인 balance() 메서드로 __balance 를 돌려주기
# 3) acc.balance 를 괄호 없이 출력해 보기
# 4) acc.balance = 999999 를 try/except AttributeError 로 실행해 에러 확인
class BankAccount:
    def __init__(self, owner, balance=0):
        self.owner = owner
        self.__balance = balance

    @property
    def balance(self):
        return self.__balance


acc = BankAccount("하늘", 10000)
print("잔액:", acc.balance)   # 괄호 없이 속성처럼 읽기

try:
    acc.balance = 999999      # setter 가 없으므로 읽기 전용
except AttributeError as e:
    print("❌ 변경 불가:", e)

---

## 5. 📊 세 가지 접근 제어 한눈에 비교

| 표기 | 이름 | 의미 | 밖에서 접근 | 자식 클래스에서 | 언제 쓰나 |
|---|---|---|---|---|---|
| `name` | **public** | 누구나 사용 | ✅ 됨 | ✅ 됨 | 상품명, 닉네임 등 공개 정보 |
| `_name` | **protected** | "내부용입니다" (약속) | ⚠️ 되지만 하면 안 됨 | ✅ 됨 | 자식에게 물려줄 내부 데이터 |
| `__name` | **private** | "이 클래스만" (이름 바뀜) | ❌ AttributeError | ❌ 안 됨 | 잔액, API 키, 비밀번호 |

### 🧠 선택 기준 3단계

1. 밖에서 알아도 되고 바뀌어도 괜찮다 → **public**
2. 자식 클래스는 써야 하지만 외부인은 안 된다 → **protected (`_`)**
3. 아무도 직접 건드리면 안 된다 → **private (`__`)** + getter/setter

---

## 6. 자주 하는 실수 TOP 5

| 증상 | 원인 | 해결 |
|---|---|---|
| `AttributeError: 'A' object has no attribute '__x'` | private 을 밖에서 접근 | getter 메서드를 만들어 사용 |
| `__init__` 을 private 으로 착각 | 앞뒤로 밑줄 2개 = **특별 메서드** | private 은 **앞에만** 2개 |
| `_` 를 쓰면 막힐 거라 기대 | `_` 는 **약속**일 뿐 | 정말 막으려면 `__` 사용 |
| `obj.__x = 5` 로 바꿨는데 반영 안 됨 | 새로운 public 속성이 추가된 것 | setter 메서드로 변경 |
| 모든 속성을 `__` 로 감쌈 | 과도한 은닉 → 코드가 불편해짐 | 공개해도 되는 건 public 으로 |

In [ ]:
# ✍️ 실수 재현 실습
# 1) Counter 클래스: __init__ 에서 __count = 0, get_count() 로 돌려주기
# 2) 객체를 만든 뒤 c.__count = 100 으로 바꿔 보기
# 3) get_count() 와 c.__dict__ 를 출력해, 왜 반영되지 않는지 설명해 보기
class Counter:
    def __init__(self):
        self.__count = 0

    def get_count(self):
        return self.__count


c = Counter()
c.__count = 100

print("get_count():", c.get_count())   # 0
print("__dict__:", c.__dict__)         # {'_Counter__count': 0, '__count': 100}

# 📌 내 설명:
#    클래스 안의 self.__count 는 _Counter__count 로 이름이 바뀌어 저장된다.
#    클래스 밖에서 쓴 c.__count = 100 은 이름 맹글링이 일어나지 않아서
#    '__count' 라는 새로운 public 속성이 하나 더 생겼을 뿐, 진짜 값(_Counter__count)은 그대로 0 이다.

---

## 7. 🛠️ 단계별 실습 과제

### 실습 1 안전한 학생 클래스

**요구사항**

- `Student` 클래스: `name` 은 **public**, 점수는 **private** (`__score`)
- `get_score()`: 점수 리턴
- `set_score(value)`: 0~100 범위만 허용. 범위를 벗어나면 `"⚠️ 점수는 0~100 사이여야 합니다"` 출력 후 변경하지 않기
- `is_pass()`: 60점 이상이면 `True`

In [ ]:
# ✍️ 실습 1: 빈칸을 채워 완성하세요
class Student:
    def __init__(self, name, score=0):
        self.name = name       # public
        self.__score = 0       # private
        self.set_score(score)  # 생성할 때도 검사를 거치도록

    def get_score(self):
        return self.__score

    def set_score(self, value):
        if not 0 <= value <= 100:
            print("⚠️ 점수는 0~100 사이여야 합니다")
            return
        self.__score = value

    def is_pass(self):
        return self.__score >= 60


s = Student("하늘", 85)
print(s.name, s.get_score(), "합격" if s.is_pass() else "불합격")

s.set_score(150)   # 범위 밖 → 경고, 변경 안 됨
print("점수:", s.get_score())   # 85

s.set_score(40)
print(s.name, s.get_score(), "합격" if s.is_pass() else "불합격")

### 실습 2  API 키를 숨기는 클라이언트 🔑

>

**요구사항**

- 클래스 이름: `ApiClient`
- `__init__(self, service_name, api_key)`
  - `service_name` 은 **public**, `api_key` 는 **private** (`__api_key`)
  - 호출 횟수 `_call_count` 는 **protected** (자식 클래스가 쓸 수 있도록), 0 으로 시작
- `get_masked_key()`: 키의 **앞 4글자만 보여 주고 나머지는 `*`** 로 가린 문자열 리턴
  - 예: `"sk-test-1234"` → `"sk-t********"`
- `ask(self, prompt)`: `_call_count` 1 증가 후 `"[서비스명] 답변: '프롬프트'"` 리턴
- `get_call_count()`: 호출 횟수 리턴
- 밖에서 `client.__api_key` 접근 시 에러가 나는지 확인하기

In [ ]:
# ✍️ 실습 2: 요구사항을 보고 ApiClient 클래스를 완성하세요
class ApiClient:
    def __init__(self, service_name, api_key):
        self.service_name = service_name   # public
        self.__api_key = api_key           # private
        self._call_count = 0               # protected

    def get_masked_key(self):
        return self.__api_key[:4] + "*" * (len(self.__api_key) - 4)

    def ask(self, prompt):
        self._call_count += 1
        return f"[{self.service_name}] 답변: '{prompt}'"

    def get_call_count(self):
        return self._call_count


# 테스트 코드 (수정 금지)
client = ApiClient("FakeGPT", "sk-test-1234")
print("서비스:", client.service_name)
print("키(가림):", client.get_masked_key())     # 기대값: sk-t********
print(client.ask("안녕?"))
print(client.ask("파이썬이 뭐야?"))
print("호출 횟수:", client.get_call_count())     # 기대값: 2

try:
    print(client.__api_key)
except AttributeError as e:
    print("❌ 키 접근 차단:", e)

### 실습 3 [중급 · 포트폴리오용 · 25분] 챗봇 v3 — 기록을 보호하기

> 지난 차시 챗봇에 **접근 제어**를 적용합니다. 대화 기록이 밖에서 조작되면 안 되니까요.

**요구사항**

- `ChatBot` 클래스
  - `name` 은 public
  - 대화 기록 `__history` 는 **private** (리스트)
  - 최대 저장 개수 `_max_history` 는 **protected**, 기본값 3
- `say(message)`: 기록에 추가. 단, 기록이 `_max_history` 를 넘으면 **가장 오래된 것부터 삭제**
- `get_history()`: 기록 **복사본**을 리턴 (원본이 밖에서 수정되지 않도록! 힌트: `리스트[:]`)
- `count()`: 현재 기록 개수 리턴
- 테스트: 4번 말한 뒤 기록이 3개만 남는지, `get_history()` 결과를 밖에서 수정해도 원본이 안전한지 확인

In [ ]:
# ✍️ 실습 3: ChatBot 클래스를 직접 작성하세요 (골격만 제공)
class ChatBot:
    def __init__(self, name, max_history=3):
        self.name = name                  # public
        self.__history = []               # private
        self._max_history = max_history   # protected

    def say(self, message):
        self.__history.append(message)
        if len(self.__history) > self._max_history:
            self.__history.pop(0)         # 가장 오래된 기록 삭제
        print(f"[{self.name}] 기억할게요 → '{message}'")

    def get_history(self):
        return self.__history[:]          # 복사본을 돌려줘서 원본 보호

    def count(self):
        return len(self.__history)


# 테스트 코드 (수정 금지)
bot = ChatBot("코코")
bot.say("안녕")
bot.say("날씨 알려줘")
bot.say("노래 추천해줘")
bot.say("파이썬 재밌다")

print("기록:", bot.get_history())
print("개수:", bot.count())              # 기대값: 3

copied = bot.get_history()
copied.clear()
print("조작 시도 후 원본:", bot.get_history())   # 기대값: 기록 3개 그대로

---

## 8. 📝 이해도 확인 퀴즈

**Q1.** `self.name`, `self._name`, `self.__name` 의 차이를 각각 한 줄로 설명해 보세요.
- 📌 내 답:
  - `self.name` : public. 클래스 밖에서도 자유롭게 읽고 쓸 수 있다.
  - `self._name` : protected. "내부·자식 클래스 전용"이라는 약속일 뿐, 밖에서 접근해도 에러는 나지 않는다.
  - `self.__name` : private. 이름 맹글링으로 `_클래스이름__name` 으로 바뀌어 밖에서 `obj.__name` 으로 접근하면 `AttributeError` 가 난다.

**Q2.** `__balance` 는 파이썬 내부에서 어떤 이름으로 바뀌나요?
- 📌 내 답: `_클래스이름__balance` (예: `BankAccount` 클래스라면 `_BankAccount__balance`)

**Q3.** 편의점 비유에서 **창고**에 해당하는 표기는 무엇인가요?
- 📌 내 답: `_name` (protected, 밑줄 1개)

**Q4.** 아래 코드의 출력을 **먼저 예측**한 뒤 다음 셀에서 확인하세요.
- 📌 내 예측:
  ```
  1 2 3
  1 2
  c 는 직접 못 봄
  3
  ```
  (`show()` 는 클래스 안이라 `__c` 접근 가능, 밖에서 `box.__c` 는 에러, 맹글링된 이름 `box._Box__c` 로는 접근됨)

**Q5.** API 키는 public / protected / private 중 어디에 저장해야 하며, 왜 그런가요?
- 📌 내 답: **private** (`__api_key`). API 키가 밖으로 새면 남이 내 계정으로 요청을 보내 요금이 나가거나 악용될 수 있다. 그래서 밖에서도, 자식 클래스에서도 직접 건드리지 못하게 가장 강하게 숨기고, 꼭 보여 줘야 하면 `get_masked_key()` 처럼 가린 값만 내보낸다.

In [ ]:
class Box:
    def __init__(self):
        self.a = 1
        self._b = 2
        self.__c = 3

    def show(self):
        print(self.a, self._b, self.__c)

box = Box()
box.show()
print(box.a, box._b)

try:
    print(box.__c)
except AttributeError:
    print("c 는 직접 못 봄")

print(box._Box__c)